In [1]:
from transformers import AutoConfig, AutoTokenizer
MODEL_ID = "allenai/OLMoE-1B-7B-0125-Instruct"

In [3]:
cfg = AutoConfig.from_pretrained(MODEL_ID)
print(type(cfg))
print(cfg)

<class 'transformers.models.olmoe.configuration_olmoe.OlmoeConfig'>
OlmoeConfig {
  "architectures": [
    "OlmoeForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": null,
  "clip_qkv": null,
  "dtype": "bfloat16",
  "eos_token_id": 50279,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 1024,
  "max_position_embeddings": 4096,
  "model_type": "olmoe",
  "norm_topk_prob": false,
  "num_attention_heads": 16,
  "num_experts": 64,
  "num_experts_per_tok": 8,
  "num_hidden_layers": 16,
  "num_key_value_heads": 16,
  "output_router_logits": false,
  "pad_token_id": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "rope_theta": 10000.0,
    "rope_type": "default"
  },
  "router_aux_loss_coef": 0.01,
  "tie_word_embeddings": false,
  "transformers_version": "5.23.0",
  "use_cache": false,
  "vocab_size": 50304
}



In [4]:
print("blocks:          ", cfg.num_hidden_layers)
print("experts/block:   ", cfg.num_experts)
print("experts/token:   ", cfg.num_experts_per_tok)
print("hidden size:     ", cfg.hidden_size)
print("expert inner:    ", cfg.intermediate_size)
print("norm_topk_prob:  ", cfg.norm_topk_prob)

per_expert = 3 * cfg.hidden_size * cfg.intermediate_size   # gate_proj + up_proj + down_proj
router_row = cfg.hidden_size                                # the new 65th router row
per_block  = per_expert + router_row
total      = per_block * cfg.num_hidden_layers

print(f"one expert:        {per_expert:,}")
print(f"+ router row:      {per_block:,}")
print(f"all 16 blocks:     {total:,}")
print(f"fraction of model: {total / 6.9e9:.2%}")

blocks:           16
experts/block:    64
experts/token:    8
hidden size:      2048
expert inner:     1024
norm_topk_prob:   False
one expert:        6,291,456
+ router row:      6,293,504
all 16 blocks:     100,696,064
fraction of model: 1.46%


In [8]:
import torch
torch.manual_seed(0)

hidden_size = 4
num_expert = 6
top_kept = 2

x = torch.randn(hidden_size)
W = torch.randn(num_expert, hidden_size)
scores = W @ x
softmax_score = torch.softmax(score, dim = 0)
top_shares, top_ids = torch.topk(softmax_score, top_kept)

print("scores:    ", scores)
print("shares:    ", softmax_score, " sum =", softmax_score.sum().item())
print("chosen:    ", top_ids, " with shares", top_shares)
print("kept share:", top_shares.sum().item())  

scores:     tensor([-1.8636,  1.6440,  0.2598, -0.5008,  0.2708,  2.4287])
shares:     tensor([0.0078, 0.2602, 0.0652, 0.0305, 0.0659, 0.5704])  sum = 1.0
chosen:     tensor([5, 1])  with shares tensor([0.5704, 0.2602])
kept share: 0.8306131362915039


In [11]:
new_row = torch.full((1, hidden_size), -10.0)
W65 = torch.concat([W, new_row], dim = 0)

shares7 = torch.softmax(W65 @ x, dim=0)
print("old 6 shares, before:", softmax_score)
print("old 6 shares, after: ", shares7[:6])
print("new expert's share:  ", shares7[6])

old 6 shares, before: tensor([0.0078, 0.2602, 0.0652, 0.0305, 0.0659, 0.5704])
old 6 shares, after:  tensor([0.0027, 0.0900, 0.0225, 0.0105, 0.0228, 0.1972])
new expert's share:   tensor(0.6542)
